In [17]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
import pandas as pd

from src.data.loaders import load_yambda_lag, split_and_reindex


In [19]:
config = {
    "ml-20m": {
        "max_seq_len": 200,
        "test_quantile": 0.1,
        "gap_seconds": 0,
        "interactions_path": "/Users/hadhad/HSE/нейро реки/проект/Actions-Speak-Louder-than-Words-research/tmp/ml-20m.zip", # https://grouplens.org/datasets/movielens/20m/
        "col_mapping": {
            "userid": "user_id",
            "movieid": "item_id",
            "rating": "feedback",
            "timestamp": "timestamp",
        },
    },
    "yambda": {
        "max_seq_len": 200,
        "test_quantile": 0.1,
        "gap_seconds": 0,
        "interactions_path": None,
        "col_mapping": {
            "uid": "user_id",
            "item_id": "item_id",
            "timestamp": "timestamp",
        },
    },
    "yambda-retrieval": {
        "max_seq_len": 100,
        "test_quantile": 0.1,
        "gap_seconds": 0,
        "interactions_path": "/Users/hadhad/HSE/нейро реки/проект/Actions-Speak-Louder-than-Words-research/tmp/yambda-10m.parquet",
        "col_mapping": {
            "uid": "user_id",
            "item_id": "item_id",
            "timestamp": "timestamp",
        },
    },
    "amzn-books": {
        "max_seq_len": 50,
        "test_quantile": 0.1,
        "gap_seconds": 0,
        "interactions_path": "/Users/hadhad/HSE/нейро реки/проект/Actions-Speak-Louder-than-Words-research/tmp/ratings_Books.csv.gz", # https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/benchmark/0core/rating_only/Books.csv.gz
        "col_mapping": {
            "user_id": "user_id",
            "parent_asin": "item_id",
            "rating": "feedback",
            "timestamp": "timestamp",
        },
    },
}


In [ ]:
yambda_df = load_yambda_lag(interactions_path=None, config=config["yambda"])

In [ ]:
yambda_train, yambda_test, yambda_data_description = split_and_reindex(yambda_df, config=config["yambda"])

In [ ]:
yambda_train

In [ ]:
datasets = {
    # "ml-20m": {
    #     "train": ml_train,
    #     "test": ml_test,
    #     "desc": ml_data_description,
    # },
    "yambda": {
        "train": yambda_train,
        "test": yambda_test,
        "desc": yambda_data_description,
    },
    # "amzn-books": {
    #     "train": amzn_train,
    #     "test": amzn_test,
    #     "desc": amzn_data_description,
    # },
}

In [ ]:
for ds in datasets.values():
    display(ds['train'].head())

In [ ]:
# статистики по датасетам

stats = []
for name, d in datasets.items():
    train, test, desc = d["train"], d["test"], d["desc"]
    n_train = len(train)
    n_test = len(test)
    n_total = n_train + n_test
    stats.append({
        "dataset": name,
        "n_train": n_train,
        "n_test": n_test,
        "n_total": n_total,
        "train_share": round(n_train / n_total, 3),
        "test_share": round(n_test  / n_total, 3),
        "n_train_users": train[desc["users"]].nunique(),
        "n_test_users": test[desc["users"]].nunique(),
        "n_items": train[desc["items"]].nunique(),
    })

pd.DataFrame(stats)

In [ ]:
import importlib

import torch

import src.data.hstu_typed_dataset as hstu_typed_dataset_module
import src.training.hstu.hstu_typed_pipeline as hstu_typed_pipeline_module

importlib.reload(hstu_typed_dataset_module)
importlib.reload(hstu_typed_pipeline_module)

from src.data.hstu_typed_dataset import (
    YAMBDA_HASH_CARDINALITY,
    YAMBDA_LABEL_COLUMNS,
    YAMBDA_MULTIVALENT_FEATURES_CONFIG,
    YAMBDA_SPARSE_FEATURES_CONFIG,
    add_yambda_multihash_feature_tokens,
    yambda_multihash_token_schema,
)
from src.training.hstu.hstu_typed_pipeline import (
    TypedHSTUExperimentConfig,
    build_typed_hstu_dataloaders,
    build_typed_hstu_model,
    evaluate_typed_hstu,
    train_typed_hstu,
)


## Typed HSTU Yambda comparison preset

Параметры ниже согласованы с `deeprecsys_project/examples/example.ipynb`: hash-trick `CARDINALITY = 65536`, seeds `(55, 66)` для `artist_ids`, `(77, 88)` для `album_ids`, `lr = 1e-3`, `epochs = 3`, hidden width около `model_dim = 64`, dropout `0.1`.

Ranking-постановка HSTU: каждый supervised example строится как `history + candidate event`; supervised head предсказывает `is_like` и `is_full_play` на candidate token. Dense lag features и raw `uid` не добавляются: typed HSTU здесь сравнивается как последовательная модель на item/category tokens.


In [ ]:
CARDINALITY = YAMBDA_HASH_CARDINALITY

COMPARISON_EPOCHS = 3
COMPARISON_LR = 1e-3
COMPARISON_EMBEDDING_DIM = 64
COMPARISON_NUM_HEADS = 4
COMPARISON_HEAD_DIM = 16
COMPARISON_DROPOUT = 0.1
MAX_HSTU_EVENTS = 64

yambda_train_typed, yambda_test_typed = add_yambda_multihash_feature_tokens(
    train=yambda_train,
    test=yambda_test,
)
yambda_schema = yambda_multihash_token_schema(
    train=yambda_train_typed,
    test=yambda_test_typed,
)

{
    "hash_cardinality": CARDINALITY,
    "sparse_hash_seeds": YAMBDA_SPARSE_FEATURES_CONFIG,
    "multivalent_hash_seeds": YAMBDA_MULTIVALENT_FEATURES_CONFIG,
    "label_columns": YAMBDA_LABEL_COLUMNS,
    "comparison_epochs": COMPARISON_EPOCHS,
    "comparison_lr": COMPARISON_LR,
    "comparison_embedding_dim": COMPARISON_EMBEDDING_DIM,
    "comparison_num_heads": COMPARISON_NUM_HEADS,
    "comparison_head_dim": COMPARISON_HEAD_DIM,
    "comparison_dropout": COMPARISON_DROPOUT,
    "max_history_events": MAX_HSTU_EVENTS,
    "tokens_per_event_upper_bound": yambda_schema.tokens_per_event,
    "max_token_seq_len_upper_bound": (MAX_HSTU_EVENTS + 1) * yambda_schema.tokens_per_event,
    "num_token_types": yambda_schema.num_token_types,
    "feature_vocab_sizes": yambda_schema.feature_vocab_sizes,
    "sample_artist_tokens": yambda_train_typed["artist_ids_token"].head().tolist(),
    "sample_album_tokens": yambda_train_typed["album_ids_token"].head().tolist(),
}


In [ ]:
hstu_config = TypedHSTUExperimentConfig(
    max_events_len=MAX_HSTU_EVENTS,
    train_batch_size=256,
    eval_batch_size=256,
    num_epochs=COMPARISON_EPOCHS,
    learning_rate=COMPARISON_LR,
    weight_decay=0.0,
    output_size=2,
    label_columns=YAMBDA_LABEL_COLUMNS,
    embedding_dim=COMPARISON_EMBEDDING_DIM,
    num_blocks=2,
    num_heads=COMPARISON_NUM_HEADS,
    linear_dim=COMPARISON_HEAD_DIM,
    attention_dim=COMPARISON_HEAD_DIM,
    input_dropout_rate=COMPARISON_DROPOUT,
    linear_dropout_rate=COMPARISON_DROPOUT,
    attn_dropout_rate=COMPARISON_DROPOUT,
    enable_relative_attention_bias=True,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

(
    hstu_train_loader,
    hstu_eval_loader,
    hstu_train_dataset,
    hstu_eval_dataset,
    hstu_test_events,
) = build_typed_hstu_dataloaders(
    train=yambda_train_typed,
    test=yambda_test_typed,
    schema=yambda_schema,
    max_events_len=hstu_config.max_events_len,
    train_batch_size=hstu_config.train_batch_size,
    eval_batch_size=hstu_config.eval_batch_size,
    user_col=yambda_data_description["users"],
    item_col=yambda_data_description["items"],
    time_col=yambda_data_description["timestamp"],
    label_columns=hstu_config.label_columns,
    num_workers=hstu_config.num_workers,
)

train_supervision_events = sum(len(sample["targets"]) for sample in hstu_train_dataset)
eval_supervision_events = sum(len(sample["targets"]) for sample in hstu_eval_dataset)
raw_train_supervision_events = int(
    yambda_train_typed.groupby(yambda_data_description["users"]).size().sub(1).clip(lower=0).sum()
)
raw_eval_supervision_events = sum(len(user_events) for user_events in hstu_test_events.values())

{
    "train_chunk_samples": len(hstu_train_dataset),
    "eval_chunk_samples": len(hstu_eval_dataset),
    "train_supervision_events": train_supervision_events,
    "eval_supervision_events": eval_supervision_events,
    "raw_train_supervision_events": raw_train_supervision_events,
    "raw_eval_supervision_events": raw_eval_supervision_events,
    "target_users": len(hstu_test_events),
    "max_history_events": hstu_config.max_events_len,
    "max_event_seq_len": hstu_config.max_events_len + 1,
    "max_token_seq_len": (hstu_config.max_events_len + 1) * yambda_schema.tokens_per_event,
    "train_batch_size_chunks": hstu_config.train_batch_size,
    "eval_batch_size_chunks": hstu_config.eval_batch_size,
}


In [ ]:
hstu_model = build_typed_hstu_model(
    num_items=yambda_data_description["n_items"],
    schema=yambda_schema,
    experiment_config=hstu_config,
)

hstu_model.to(hstu_config.device)
hstu_optimizer = torch.optim.AdamW(
    hstu_model.parameters(),
    lr=hstu_config.learning_rate,
    weight_decay=hstu_config.weight_decay,
)


In [ ]:
module_rows = []
for name, module in hstu_model.named_children():
    params = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    module_rows.append({"module": name, "params": params, "trainable": trainable})

params_df = pd.DataFrame(module_rows)
{
    "by_top_level_module": params_df,
    "params_sum": int(params_df["params"].sum()),
    "trainable_sum": int(params_df["trainable"].sum()),
}


In [ ]:
losses, eval_history = train_typed_hstu(
    model=hstu_model,
    train_loader=hstu_train_loader,
    optimizer=hstu_optimizer,
    num_epochs=hstu_config.num_epochs,
    device=hstu_config.device,
    eval_loader=hstu_eval_loader,
    label_columns=hstu_config.label_columns,
)
losses, eval_history


In [ ]:
eval_history_df = pd.DataFrame(eval_history)
eval_history_df.insert(0, "epoch", range(1, len(eval_history_df) + 1))
eval_history_df


## Normalized Entropy

NE считается на уровне событий, а не на уровне `Dataset` samples. После перехода на chunk-сэмплы один sample может содержать несколько `supervision_token_positions`, поэтому количество samples уменьшается, но `labels` и `logits` в `evaluate_typed_hstu` остаются event-level.

Для сопоставимости с прежней typed-постановкой supervised head применяется к токенам candidate events внутри chunk. Causal attention не даёт событию видеть будущие candidate events в том же chunk, но даёт видеть собственные item/category tokens, как и раньше в формате `history + candidate event`. Поэтому normalized entropy меняется только из-за нового батчинга/контекста внутри chunk и стохастики обучения, а не из-за смены формулы метрики.


In [ ]:
hstu_metrics, hstu_eval_outputs = evaluate_typed_hstu(
    model=hstu_model,
    eval_loader=hstu_eval_loader,
    device=hstu_config.device,
    label_columns=hstu_config.label_columns,
)

{
    **hstu_metrics,
    "eval_events": int(hstu_eval_outputs["labels"].shape[0]),
    "eval_chunk_samples": len(hstu_eval_dataset),
}


In [ ]:
pd.DataFrame(
    {
        "model": ["typed hstu"],
        "e_task": [hstu_metrics["ne_e_task"]],
        "c_task": [hstu_metrics["ne_c_task"]],
    }
)
